# Chunking Strategies for Document Processing
**Author:** Juan Esteban Agudelo Ortiz  
**Email:** juan.es.agor@gmail.com

---

This notebook implements and compares three chunking strategies for
splitting extracted text into fragments suitable for downstream language
model processing. The input is the text field of an `IngestedDocument`
produced by the ingestion pipeline notebook.

Chunking is necessary because language models have a finite context window:
the maximum number of tokens they can process in a single call. A token is
approximately 0.75 words in English. For a model with a context window of
$N$ tokens, any input exceeding $N$ tokens must be split into chunks before
processing.

### Limitations
1. Fixed-size chunking may split sentences mid-way, breaking coherence at chunk boundaries.
2. Section-based chunking requires consistent section headers; degraded by poor PDF extraction.
3. Semantic chunking is computationally more expensive than the other two strategies.
4. No single strategy is optimal for all document types; the demo includes qualitative comparison.
5. Token estimates assume general English text; technical chemistry terminology 
   and molecular formulas tokenize less efficiently, reducing effective chunk size.

## 0. Install dependencies

In [17]:
# Run only once
# !pip install llama-index-core llama-index-embeddings-huggingface nltk numpy

## 1. Imports and configuration

In [18]:
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import nltk
import numpy as np
from llama_index.core.node_parser import (
    SentenceSplitter,
    SemanticSplitterNodeParser,
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Document

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# --- Directory setup ---
BASE_DIR    = Path("..")
UPLOADS_DIR = BASE_DIR / "data" / "uploads"
OUT_DIR     = BASE_DIR / "outputs"

UPLOADS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Reproduce IngestedDocument from notebook 1 ---
from enum import Enum

class InputType(Enum):
    PDF           = "pdf"
    HANDWRITTEN   = "handwritten_image"
    REFERENCE_IMG = "reference_image"
    PLAIN_TEXT    = "plain_text"

@dataclass
class IngestedDocument:
    input_type       : InputType
    text             : str
    reference_images : list = field(default_factory=list)
    source_path      : Optional[Path] = None

print(f"Uploads dir : {UPLOADS_DIR.resolve()}")
print(f"Outputs dir : {OUT_DIR.resolve()}")
print("Configuration ready ✓")

Uploads dir : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/data/uploads
Outputs dir : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/outputs
Configuration ready ✓


## 2. Why chunking matters

A language model with context window of $N$ tokens cannot process a document
of $M > N$ tokens in a single call. The document must be split into chunks
of size $c \leq N$ tokens before processing.

The choice of chunk size $c$ and overlap $o$ involves a tradeoff:

- Large $c$: more context per chunk, but fewer chunks fit in the model's
  context window simultaneously during retrieval.
- Small $c$: more precise retrieval, but chunks may lack sufficient context
  to answer a question independently.

For a document of $M$ tokens, chunk size $c$, and overlap $o$, the number
of chunks is approximately:

$$n_{chunks} \approx \left\lceil \frac{M - o}{c - o} \right\rceil$$

Overlap $o$ ensures that information at chunk boundaries is not lost: each
chunk shares $o$ tokens with the next chunk.

## 3. Fixed-size chunking with overlap

The simplest chunking strategy. The text is split into chunks of exactly
$c$ tokens with an overlap of $o$ tokens between consecutive chunks.

LlamaIndex `SentenceSplitter` respects sentence boundaries: instead of
cutting at exactly token $c$, it cuts at the sentence boundary closest
to $c$. This preserves coherence at chunk boundaries at the cost of
slightly variable chunk sizes.

In [19]:
def fixed_size_chunking(
    doc: IngestedDocument,
    chunk_size: int = 512,
    chunk_overlap: int = 64,
) -> list[dict]:
    """
    Split document text into fixed-size chunks with overlap.

    Parameters
    ----------
    doc : IngestedDocument
        Document to chunk. Must have non-empty text field.
    chunk_size : int
        Target chunk size in tokens. Default 512.
    chunk_overlap : int
        Number of overlapping tokens between consecutive chunks. Default 64.

    Returns
    -------
    list[dict]
        List of chunks, each with keys:
        - 'text'     : str  — chunk text content
        - 'index'    : int  — chunk position in document
        - 'strategy' : str  — chunking strategy used
    """
    splitter = SentenceSplitter(
        chunk_size    = chunk_size,
        chunk_overlap = chunk_overlap,
    )

    llama_doc = Document(text=doc.text)
    nodes = splitter.get_nodes_from_documents([llama_doc])

    return [
        {
            "text"     : node.text,
            "index"    : i,
            "strategy" : "fixed_size",
        }
        for i, node in enumerate(nodes)
    ]


# Quick test
test_doc = IngestedDocument(
    input_type = InputType.PLAIN_TEXT,
    text       = "Amides are compounds derived from carboxylic acids. " * 50,
)

chunks = fixed_size_chunking(test_doc)
print(f"Chunks generated : {len(chunks)}")
print(f"First chunk size : {len(chunks[0]['text'])} chars")
print(f"\n--- First chunk preview ---")
print(chunks[0]['text'][:200])

Chunks generated : 2
First chunk size : 2027 chars

--- First chunk preview ---
Amides are compounds derived from carboxylic acids. Amides are compounds derived from carboxylic acids. Amides are compounds derived from carboxylic acids. Amides are compounds derived from carboxylic


## 4. Section-based chunking

Instead of splitting by token count, this strategy splits the document
at section boundaries detected by header patterns. Each chunk corresponds
to one logical section of the document.

For OpenStax PDFs, section headers follow patterns like:
- `21.1 Nucleophilic Acyl Substitution`
- `Why This Chapter?`
- `Key Terms`

A chunk is created for each section, from its header to the start of
the next header. This preserves the logical structure of the document
and produces chunks that are semantically self-contai

In [20]:
def section_based_chunking(
    doc: IngestedDocument,
    chunk_size: int = 512,
    chunk_overlap: int = 64,
) -> list[dict]:
    """
    Split document text into chunks based on section boundaries.
    Sections exceeding chunk_size tokens are further split using
    fixed-size chunking as fallback.

    Parameters
    ----------
    doc : IngestedDocument
        Document to chunk. Must have non-empty text field.
    chunk_size : int
        Maximum chunk size in tokens before fallback splitting.
    chunk_overlap : int
        Overlap tokens used in fallback fixed-size splitting.

    Returns
    -------
    list[dict]
        List of chunks, each with keys:
        - 'text'     : str  — chunk text content
        - 'index'    : int  — chunk position in document
        - 'strategy' : str  — chunking strategy used
    """
    # Pattern matches OpenStax section headers and page markers
    section_pattern = re.compile(
        r"(?:--- Page \d+ ---|\d+\.\d+\s+[A-Z][^\n]+|Why This Chapter\?|"
        r"Key Terms|Summary|Additional Problems)"
    )

    # Split text at section boundaries
    sections = section_pattern.split(doc.text)
    sections = [s.strip() for s in sections if s.strip()]

    splitter = SentenceSplitter(
        chunk_size    = chunk_size,
        chunk_overlap = chunk_overlap,
    )

    chunks = []
    for section in sections:
        # Estimate token count: 1 token ≈ 0.75 words
        word_count  = len(section.split())
        token_count = int(word_count / 0.75)

        if token_count <= chunk_size:
            chunks.append(section)
        else:
            # Fallback: fixed-size chunking on long sections
            llama_doc    = Document(text=section)
            nodes        = splitter.get_nodes_from_documents([llama_doc])
            chunks.extend([node.text for node in nodes])

    return [
        {
            "text"     : chunk,
            "index"    : i,
            "strategy" : "section_based",
        }
        for i, chunk in enumerate(chunks)
    ]


# Quick test
test_doc = IngestedDocument(
    input_type = InputType.PLAIN_TEXT,
    text       = "--- Page 1 ---\nAmides are derived from carboxylic acids. " * 20
                 + "\n--- Page 2 ---\nEsters are formed by reaction with alcohols. " * 20,
)

chunks = section_based_chunking(test_doc)
print(f"Chunks generated : {len(chunks)}")
print(f"\n--- First chunk preview ---")
print(chunks[0]['text'][:200])

Chunks generated : 40

--- First chunk preview ---
Amides are derived from carboxylic acids.


## 5. Semantic chunking

Instead of splitting by token count or section boundaries, semantic
chunking groups sentences by meaning similarity. Consecutive sentences
with high semantic similarity stay in the same chunk; a split occurs
when similarity drops below a threshold $\tau$.

Similarity between consecutive sentences $s_i$ and $s_{i+1}$ is measured
by cosine similarity between their embedding vectors $\mathbf{e}_i$ and
$\mathbf{e}_{i+1}$:

$$\text{sim}(s_i, s_{i+1}) = \frac{\mathbf{e}_i \cdot \mathbf{e}_{i+1}}{\|\mathbf{e}_i\| \|\mathbf{e}_{i+1}\|}$$

A split is inserted when $\text{sim}(s_i, s_{i+1}) < \tau$. Lower $\tau$
produces fewer, larger chunks; higher $\tau$ produces more, smaller chunks.

This strategy is computationally more expensive than the others because
it requires embedding every sentence before chunking.

In [21]:
def semantic_chunking(
    doc: IngestedDocument,
    embed_model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    breakpoint_percentile: int = 95,
) -> list[dict]:
    """
    Split document text into chunks based on semantic similarity between
    consecutive sentences. A split occurs when similarity drops below a
    threshold derived from the breakpoint_percentile of all pairwise
    similarities.

    Parameters
    ----------
    doc : IngestedDocument
        Document to chunk. Must have non-empty text field.
    embed_model_name : str
        HuggingFace model name for sentence embeddings.
    breakpoint_percentile : int
        Percentile of similarity scores used as split threshold.
        Higher values produce more, smaller chunks.

    Returns
    -------
    list[dict]
        List of chunks, each with keys:
        - 'text'     : str  — chunk text content
        - 'index'    : int  — chunk position in document
        - 'strategy' : str  — chunking strategy used
    """
    

    device = 'cpu'
    print(f"Device: {device}")
    embed_model = HuggingFaceEmbedding(model_name=embed_model_name,
                                       device=device)

    splitter = SemanticSplitterNodeParser(
        embed_model           = embed_model,
        breakpoint_percentile_threshold = breakpoint_percentile,
    )

    llama_doc = Document(text=doc.text)
    nodes     = splitter.get_nodes_from_documents([llama_doc])

    return [
        {
            "text"     : node.text,
            "index"    : i,
            "strategy" : "semantic",
        }
        for i, node in enumerate(nodes)
    ]


print("semantic_chunking defined ✓")
print("Note: first call downloads the embedding model (~90MB)")

semantic_chunking defined ✓
Note: first call downloads the embedding model (~90MB)


## 6. Comparison of strategies

The three strategies are compared on the same document using three
qualitative criteria:

| Criterion | Description |
|---|---|
| Number of chunks | Total chunks produced |
| Mean chunk size | Average number of characters per chunk |
| Size variance | Standard deviation of chunk sizes |

Low size variance indicates consistent chunks. High variance indicates
that some chunks are much larger or smaller than the mean, which can
produce inconsistent retrieval quality.

In [22]:
def compare_chunking_strategies(
    doc: IngestedDocument,
    chunk_size: int = 512,
    chunk_overlap: int = 64,
) -> dict:
    """
    Compare fixed-size, section-based, and semantic chunking strategies
    on the same document.

    Parameters
    ----------
    doc : IngestedDocument
        Document to chunk.
    chunk_size : int
        Target chunk size in tokens for fixed-size and section-based.
    chunk_overlap : int
        Overlap tokens for fixed-size and section-based strategies.

    Returns
    -------
    dict
        Comparison metrics for each strategy.
    """
    results = {}

    for strategy_name, strategy_fn in [
        ("fixed_size",    lambda d: fixed_size_chunking(d, chunk_size, chunk_overlap)),
        ("section_based", lambda d: section_based_chunking(d, chunk_size, chunk_overlap)),
        ("semantic",      lambda d: semantic_chunking(d)),
    ]:
        chunks     = strategy_fn(doc)
        sizes      = [len(c["text"]) for c in chunks]
        results[strategy_name] = {
            "n_chunks"   : len(chunks),
            "mean_size"  : float(np.mean(sizes)),
            "std_size"   : float(np.std(sizes)),
            "min_size"   : min(sizes),
            "max_size"   : max(sizes),
        }

    # Print comparison table
    print(f"{'Strategy':<16} {'Chunks':>8} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
    print("-" * 58)
    for name, metrics in results.items():
        print(
            f"{name:<16} "
            f"{metrics['n_chunks']:>8} "
            f"{metrics['mean_size']:>8.0f} "
            f"{metrics['std_size']:>8.0f} "
            f"{metrics['min_size']:>8} "
            f"{metrics['max_size']:>8}"
        )

    return results


print("compare_chunking_strategies defined ✓")

compare_chunking_strategies defined ✓


## 7. Demo — Chunking OpenStax chapter 21

This demo applies the three chunking strategies to the carboxylic acid
derivatives chapter from OpenStax Organic Chemistry and compares their
outputs qualitatively.

The semantic chunking step may take several minutes on CPU. The fixed-size
and section-based strategies complete in under a second.

### Getting the PDF
Download the full book from
```text
https://openstax.org/details/books/organic-chemistry
```
Then extract chapter 21 (pages 741 to 792) using PyMuPDF:

```python
import fitz
doc = fitz.open("openstax_organic_chemistry.pdf")
sub = fitz.open()
sub.insert_pdf(doc, from_page=740, to_page=791)
sub.save("data/uploads/openstax_ch21_carboxylic_acid_derivatives.pdf")
```

Note: PyMuPDF uses zero-based page indexing, so page 741 corresponds to index 740.

In [23]:
import fitz

def extract_text_from_pdf(pdf_path: Path, min_block_chars: int = 20) -> str:
    """
    Extract ordered text from a PDF using block-based extraction.
    Reproduced from the ingestion pipeline notebook.

    Parameters
    ----------
    pdf_path : Path
        Path to the PDF file.
    min_block_chars : int
        Minimum character count for a block to be included.

    Returns
    -------
    str
        Clean, ordered text extracted from all pages.
    """
    import re
    doc = fitz.open(pdf_path)
    all_text = []

    for page_num, page in enumerate(doc):
        blocks = page.get_text("blocks")
        blocks_sorted = sorted(blocks, key=lambda b: (b[1], b[0]))

        page_text = []
        for block in blocks_sorted:
            text = block[4].strip()
            if len(text) < min_block_chars:
                continue
            text = re.sub(r"\s+", " ", text)
            page_text.append(text)

        if page_text:
            all_text.append(f"--- Page {page_num + 1} ---\n" + "\n\n".join(page_text))

    doc.close()
    return "\n\n".join(all_text)


# --- Demo ---
pdf_path = UPLOADS_DIR / "openstax_ch21_carboxylic_acid_derivatives.pdf"

doc = IngestedDocument(
    input_type  = InputType.PDF,
    text        = extract_text_from_pdf(pdf_path),
    source_path = pdf_path,
)

print(f"Text extracted : {len(doc.text)} characters")
print(f"\nRunning comparison (semantic chunking may take several minutes)...")
results = compare_chunking_strategies(doc)

Text extracted : 72868 characters

Running comparison (semantic chunking may take several minutes)...
Device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Strategy           Chunks     Mean      Std      Min      Max
----------------------------------------------------------
fixed_size             42     1881      288      918     2328
section_based          96      716      621        4     2504
semantic               26     2803     2944      160    10339
